# CSV Column Name Normalizer
## Fix whitespace and special characters in column names

This notebook helps normalize CSV column names to make them Python-friendly:
- Remove leading/trailing whitespace
- Replace spaces with underscores
- Convert to lowercase (optional)
- Remove special characters

## Quick Fix: Normalize All CSVs

In [ ]:
# =============================================================================
# AUTO-INSTALL DEPENDENCIES
# Run this cell to scan the notebook and install required packages
# =============================================================================
%%calliope ask claude --format code
Scan this notebook for all Python imports and library dependencies.
Generate a shell script that:
1. Checks if each required package is installed
2. Installs any missing packages using pip
Output only the executable shell commands, no explanations.


In [ ]:
import pandas as pd
import os
from pathlib import Path

def normalize_columns(df):
    """Normalize column names to be Python-friendly"""
    df.columns = (
        df.columns
        .str.strip()  # Remove leading/trailing whitespace
        .str.replace(' ', '_')  # Replace spaces with underscores
        .str.replace('[^a-zA-Z0-9_]', '', regex=True)  # Remove special chars
    )
    return df

# Process all CSVs
csv_dir = Path('~/lab/data/CSVs').expanduser()
backup_dir = Path('~/lab/data/CSVs_backup').expanduser()

# Create backup directory
backup_dir.mkdir(exist_ok=True)

print("Normalizing CSV column names...\n")

for csv_file in csv_dir.glob('*.csv'):
    print(f"Processing: {csv_file.name}")
    
    try:
        # Backup original
        backup_path = backup_dir / csv_file.name
        if not backup_path.exists():
            import shutil
            shutil.copy2(csv_file, backup_path)
            print(f"  ✓ Backed up to: {backup_path}")
        
        # Read CSV
        df = pd.read_csv(csv_file)
        
        # Show before
        print(f"  Before: {list(df.columns)}")
        
        # Normalize
        df = normalize_columns(df)
        
        # Show after
        print(f"  After:  {list(df.columns)}")
        
        # Save normalized version
        df.to_csv(csv_file, index=False)
        print(f"  ✓ Saved normalized version\n")
        
    except Exception as e:
        print(f"  ✗ Error: {e}\n")

print("\n✅ Done! Original files backed up to ~/lab/data/CSVs_backup/")

## Preview: Check Specific File

In [ ]:
import pandas as pd

# Check the MVA file
df = pd.read_csv('~/lab/data/CSVs/MVA_Vehicle_Sales_Counts_by_Month_for_Calendar_Year_2002_through_February_2025.csv')

print("Current column names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {repr(col)}  (length: {len(col)})")

print("\nFirst few rows:")
print(df.head())

## Manual Fix: Single File
If you want to normalize just one file manually

In [ ]:
import pandas as pd

# Load file
file_path = '~/lab/data/CSVs/MVA_Vehicle_Sales_Counts_by_Month_for_Calendar_Year_2002_through_February_2025.csv'
df = pd.read_csv(file_path)

print("Before:")
print(df.columns.tolist())

# Normalize column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('[^a-zA-Z0-9_]', '', regex=True)
)

print("\nAfter:")
print(df.columns.tolist())

# Save
df.to_csv(file_path, index=False)
print("\n✓ Saved!")

## Restore from Backup
If you need to restore original files

In [ ]:
import shutil
from pathlib import Path

csv_dir = Path('~/lab/data/CSVs').expanduser()
backup_dir = Path('~/lab/data/CSVs_backup').expanduser()

if backup_dir.exists():
    print("Restoring from backup...\n")
    
    for backup_file in backup_dir.glob('*.csv'):
        target = csv_dir / backup_file.name
        shutil.copy2(backup_file, target)
        print(f"✓ Restored: {backup_file.name}")
    
    print("\n✅ All files restored!")
else:
    print("No backup directory found.")

## Alternative: Handle in Calliope Prompts

Instead of normalizing files, you can tell Calliope to handle spaces:

```python
%%calliope ask gpto --format code
Load ~/lab/data/CSVs/MVA_Vehicle_Sales_Counts_by_Month_for_Calendar_Year_2002_through_February_2025.csv

IMPORTANT: The CSV has column names with spaces and trailing whitespace:
- 'Year ' (with trailing space)
- 'Month ' (with trailing space)
- 'New'
- 'Used'
- 'Total Sales New'
- 'Total Sales Used'

First strip whitespace from column names and optionally rename them:
df.columns = df.columns.str.strip()
# or rename:
# df.columns = df.columns.str.strip().str.replace(' ', '_')

Then show the first 10 rows
```